# Acessilia Toolbox — Interactive Provider Tour

This notebook exercises the toolbox provider architecture end to end: registry loading, health checks, sample generation, explicit MinerU extraction, result inspection, and a side-by-side comparison with Docling.

**Prerequisites**
- The project virtualenv selected as this notebook's kernel (`.venv`)
- Docker Desktop running
- `docling-serve` on `http://localhost:5001` (optional, started by `docker compose up -d docling-serve`)
- The MinerU CPU image and API container:

```bash
./scripts/build-mineru-image.sh
./scripts/run-mineru.sh
./scripts/run-mineru.sh status
```

The lightweight image uses MinerU's CPU `pipeline` backend. First extraction downloads model files into the persistent `mineru-models` Docker volume and can take several minutes.

## 1. Setup

Load the registries exactly the way the application does at startup.

In [1]:
import os
from pathlib import Path

from acessilia_toolbox.core.capability import CapabilityRegistry
from acessilia_toolbox.core.provider import ProviderRegistry
from acessilia_toolbox.providers import create_adapter

# The registry expands ${VAR} placeholders when the config file is loaded, so
# endpoints must be resolved BEFORE ProviderRegistry.from_file().
#
# .env files (loaded by run-local.sh) point at Docker service names like
# `http://docling-serve:5001`, which only resolve inside the Compose network.
# Running locally, rewrite those to localhost while keeping any custom port.
_DOCKER_HOSTS = {"docling-serve": 5001, "mineru-serve": 5002, "minio": 9000}


def _local_url(env_var: str, default_port: int) -> str:
    value = os.environ.get(env_var, "")
    for host, port in _DOCKER_HOSTS.items():
        if f"//{host}:" in value:
            return f"http://localhost:{port}"
    return value or f"http://localhost:{default_port}"


os.environ["DOCLING_SERVE_URL"] = _local_url("DOCLING_SERVE_URL", 5001)
os.environ["MINERU_SERVE_URL"] = _local_url("MINERU_SERVE_URL", 5002)
os.environ["MINIO_URL"] = _local_url("MINIO_URL", 9000)
os.environ.setdefault("VALKEY_URL", "redis://localhost:6379")

# Locate the repo root by walking up until providers-config.yaml is found.
ROOT = Path.cwd()
while not (ROOT / "providers-config.yaml").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("providers-config.yaml not found — run the notebook from inside the repo")
    ROOT = ROOT.parent

capabilities = CapabilityRegistry.from_directory(ROOT / "capabilities")
providers = ProviderRegistry.from_file(ROOT / "providers-config.yaml")

print(f"Repo root: {ROOT}")
print(f"Capabilities registered: {len(capabilities.ids())}")
print(f"Providers registered:    {len(providers.descriptors())}")

Repo root: /Users/akira/dados/sync/dev/acessilia-toolbox
Capabilities registered: 21
Providers registered:    17


## 2. Explore the capability catalog

Each capability is a contract: input/output schemas, semantics, and the providers that can fulfill it.

In [2]:
for capability_id in sorted(capabilities.ids()):
    manifest = capabilities.get(capability_id)
    provider_ids = ", ".join(b.id for b in manifest.providers)
    print(f"{capability_id:<32} providers: {provider_ids}")

accessibility.mark               providers: pure-python
artifact.retrieve                providers: filesystem, minio
artifact.store                   providers: filesystem, minio
code.normalize                   providers: pure-python
dataset.describe                 providers: dataset-github, dataset-huggingface
dataset.get_artifact             providers: dataset-github, dataset-huggingface
dataset.get_item                 providers: dataset-github, dataset-huggingface
dataset.list                     providers: dataset-github, dataset-huggingface
dataset.list_items               providers: dataset-github, dataset-huggingface
dataset.list_splits              providers: dataset-github, dataset-huggingface
dataset.sample                   providers: dataset-github, dataset-huggingface
dataset.sync                     providers: dataset-github, dataset-huggingface
document.layout.analyze          providers: docling, mineru-layout
document.ocr                     providers: docling-ocr, 

## 3. Inspect the provider topology

Each provider declares its transport, endpoint, media types, and timeout.

In [3]:
for d in sorted(providers.descriptors(), key=lambda d: d.id):
    endpoint = d.endpoint or "(in-process)"
    print(f"{d.id:<20} {d.transport:<10} {endpoint}  timeout={d.timeout_seconds}s")

artifact-store       s3         http://acessilia-minio:9000  timeout=600.0s
dataset-github       in_process (in-process)  timeout=600.0s
dataset-huggingface  in_process (in-process)  timeout=600.0s
docling              http       http://localhost:5001  timeout=600.0s
docling-layout       http       http://localhost:5001  timeout=600.0s
docling-math         http       http://localhost:5001  timeout=300.0s
docling-ocr          http       http://localhost:5001  timeout=600.0s
mineru               http       http://localhost:5002  timeout=600.0s
mineru-layout        http       http://localhost:5002  timeout=600.0s
mineru-ocr           http       http://localhost:5002  timeout=600.0s
minio                s3         http://acessilia-minio:9000  timeout=600.0s
pure-accessibility   in_process (in-process)  timeout=600.0s
pure-code            in_process (in-process)  timeout=600.0s
pure-math            in_process (in-process)  timeout=600.0s
pure-text            in_process (in-process)  timeout

## 4. Health check all HTTP providers

This cell reloads the registry after local endpoint normalization, instantiates each toolbox adapter, and calls its own health check. MinerU should report healthy only after `./scripts/run-mineru.sh` has completed.

In [4]:
# Re-load the registry in case env vars were set after the first load.
providers = ProviderRegistry.from_file(ROOT / "providers-config.yaml")

for d in sorted(providers.descriptors(), key=lambda d: d.id):
    if d.transport != "http" or not d.endpoint:
        continue
    adapter = create_adapter(d)
    health = adapter.health()
    status = "✅" if health.healthy else "❌"
    detail = health.version if health.healthy else (health.detail or "unreachable")
    print(f"{status} {d.id:<20} {detail}")

✅ docling              1.32.0
✅ docling-layout       1.32.0
✅ docling-math         1.32.0
✅ docling-ocr          1.32.0
✅ mineru               0.1.0
✅ mineru-layout        0.1.0
✅ mineru-ocr           0.1.0


## 5. Inspect the MinerU image and container

These checks do not mutate Docker state. A healthy API exposes `/health`, `/openapi.json`, `/file_parse`, and asynchronous task endpoints.

In [5]:
import subprocess

checks = [
    ["docker", "images", "mineru", "--format", "{{.Repository}}:{{.Tag}} {{.Size}}"],
    ["docker", "ps", "-a", "--filter", "name=mineru-serve", "--format", "{{.Names}} {{.Status}}"],
    [str(ROOT / "scripts" / "run-mineru.sh"), "status"],
]

for command in checks:
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
    output = (completed.stdout or completed.stderr).strip()
    print(f"$ {' '.join(command)}")
    print(output or f"exit={completed.returncode}")
    print()

$ docker images mineru --format {{.Repository}}:{{.Tag}} {{.Size}}
mineru:latest 10.6GB
mineru:cpu-base 10.6GB

$ docker ps -a --filter name=mineru-serve --format {{.Names}} {{.Status}}
mineru-serve Up 2 hours (healthy)

$ /Users/akira/dados/sync/dev/acessilia-toolbox/scripts/run-mineru.sh status
✅ mineru-serve healthy at http://localhost:5002



## 6. Generate a sample document

The repository ships a sample generator, so the interactive test does not depend on external files.

In [6]:
import subprocess

SAMPLE_DIR = Path("/tmp/notebook-samples")
SAMPLE_PDF = SAMPLE_DIR / "sample-simple.pdf"

if not SAMPLE_PDF.exists():
    completed = subprocess.run(
        ["python", "scripts/generate_samples.py", "--output", str(SAMPLE_DIR)],
        cwd=ROOT,
        text=True,
        capture_output=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr or completed.stdout)

if not SAMPLE_PDF.exists():
    raise FileNotFoundError(f"Sample generator did not create {SAMPLE_PDF}")

print(f"Sample PDF: {SAMPLE_PDF} ({SAMPLE_PDF.stat().st_size} bytes)")

Sample PDF: /tmp/notebook-samples/sample-simple.pdf (672 bytes)


## 7. Execute MinerU explicitly (optional slow test)

This is the only slow cell in the notebook. It performs a real synchronous extraction with the CPU `pipeline` backend. The first run may download/load models and take several minutes.

To keep the smoke test small, it uses `sample-simple.pdf` and processes only its first page. The cell initializes `mineru_result` before starting, so later cells can detect whether extraction completed.

In [7]:
from acessilia_toolbox.core.errors import ProviderExecutionError, ProviderTimeoutError

CAPABILITY = "document.structure.extract"
PROVIDER_ID = "mineru"
mineru_result = None

payload = SAMPLE_PDF.read_bytes()
descriptor = providers.resolve(CAPABILITY, PROVIDER_ID)
adapter = create_adapter(descriptor)
health = adapter.health()

if not health.healthy:
    print(
        "MinerU is offline. From the repository root run:\n"
        "  ./scripts/run-mineru.sh"
    )
else:
    print(
        "Starting MinerU CPU extraction for one page. "
        "The first run may take several minutes..."
    )
    try:
        mineru_result = adapter.execute(
            CAPABILITY,
            payload,
            filename=SAMPLE_PDF.name,
            media_type="application/pdf",
            parameters={
                "backend": "pipeline",
                "lang": "ch",
                "start_page_id": 0,
                "end_page_id": 0,
            },
        )
    except ProviderTimeoutError as exc:
        print(f"MinerU timed out: {exc}")
        print("Inspect progress with: ./scripts/run-mineru.sh logs")
    except ProviderExecutionError as exc:
        print(f"MinerU parsing failed: {exc}")
        print("Inspect details with: ./scripts/run-mineru.sh logs")
    else:
        print(f"Provider: {PROVIDER_ID}")
        print(f"Backend:  {mineru_result.backend}")
        print(f"Version:  {mineru_result.version}")
        print(f"Duration: {mineru_result.duration_ms} ms")

Starting MinerU CPU extraction for one page. The first run may take several minutes...
Provider: mineru
Backend:  mineru
Version:  0.1.0
Duration: 8241 ms


## 8. Inspect the MinerU document facade

`MineruDocument` exposes pages, text blocks, tables, formulas, pictures, full text, and the original `middle_json` payload. These are the structures consumed by downstream normalization.

In [8]:
current_result = globals().get("mineru_result")

if current_result is None:
    print(
        "No MinerU extraction result is available. "
        "Run the optional slow test in the previous section and wait for it to finish."
    )
else:
    document = current_result.document

    print(f"Pages:       {document.page_count}")
    print(f"Text blocks: {len(document.texts)}")
    print(f"Tables:      {len(document.tables)}")
    print(f"Formulas:    {len(document.formulas)}")
    print(f"Pictures:    {len(document.pictures)}")
    print()
    print("First text items:")
    for item in document.texts[:5]:
        preview = (item.text or "")[:80]
        print(
            f"  [{item.label:<18}] page={item.page_no} "
            f"bbox={item.bbox.as_tuple()} {preview!r}"
        )

    print()
    print("Full text preview:")
    print(document.full_text[:500])

Pages:       1
Text blocks: 2
Tables:      0
Formulas:    0
Pictures:    0

First text items:
  [title             ] page=0 bbox=(69.0, 61.0, 220.0, 86.0) 'Hello Toolbox'
  [text              ] page=0 bbox=(69.0, 111.0, 310.0, 125.0) 'This is a simple test document for API validation.'

Full text preview:
Hello Toolbox

This is a simple test document for API validation.


## 9. Compare MinerU and Docling

Capabilities are interchangeable: the same contract can be served by Docling or MinerU. Run both when available and compare extraction time, page count, and extracted items.

In [9]:
rows = []
for provider_id in ("docling", "mineru"):
    descriptor = providers.resolve(CAPABILITY, provider_id)
    adapter = create_adapter(descriptor)
    health = adapter.health()
    if not health.healthy:
        rows.append((provider_id, "offline", "-", "-"))
        continue

    try:
        extraction = adapter.execute(
            CAPABILITY,
            payload,
            filename=SAMPLE_PDF.name,
            media_type="application/pdf",
            parameters={"backend": "pipeline", "lang": "ch"}
            if provider_id == "mineru"
            else None,
        )
        doc = extraction.document
        pages = doc.page_count if hasattr(doc, "page_count") else doc.num_pages()
        items = len(doc.texts) if hasattr(doc, "texts") else len(list(doc.iterate_items()))
        rows.append((provider_id, f"{extraction.duration_ms} ms", pages, items))
    except Exception as exc:
        rows.append((provider_id, type(exc).__name__, "-", "-"))

print(f"{'provider':<12} {'duration/status':<22} {'pages':<8} {'items':<8}")
for row in rows:
    print(f"{row[0]:<12} {str(row[1]):<22} {str(row[2]):<8} {str(row[3]):<8}")

provider     duration/status        pages    items   
docling      20559 ms               1        2       
mineru       10621 ms               1        2       


## 10. In-process providers (no server needed)

The `pure-*` providers run inside the toolbox process with no ML runtime. This is a useful control test when HTTP providers are offline.

In [10]:
# These adapters do not require Docker or external model runtimes.
text_adapter = create_adapter(providers.get("pure-text"))
text_result = text_adapter.execute(
    "text.postprocess",
    b'{"pages": [{"page": 1, "regions": [{"text": "Hello   world"}]}]}',
    filename="text.json",
    media_type="application/json",
)
print("text.postprocess:")
print(text_result.document)

math_adapter = create_adapter(providers.get("pure-math"))
math_result = math_adapter.execute(
    "math.convert",
    b"E = mc^2",
    filename="equation.txt",
    media_type="text/plain",
    parameters={"target": "mathml"},
)
print("\nmath.convert:")
print(math_result.document)

text.postprocess:
{'pages': [{'page_number': 0, 'text': 'Hello world', 'element_count': 1}], 'full_text': 'Hello world', 'markers_applied': [], 'deduplicated_count': 0, 'overlap_removed_count': 0, 'page_count': 1}

math.convert:
{'latex': 'E = mc^2', 'mathml': '<math xmlns="http://www.w3.org/1998/Math/MathML" display="inline"><mrow><mi>E</mi><mo>&#x0003D;</mo><mi>m</mi><msup><mi>c</mi><mn>2</mn></msup></mrow></math>', 'direction': 'latex-to-mathml'}


## 11. Troubleshooting and next steps

Useful commands from the repository root:

```bash
./scripts/run-mineru.sh status        # API health
./scripts/run-mineru.sh logs          # model downloads and parse failures
./scripts/run-mineru.sh stop          # remove the local container
./scripts/build-mineru-image.sh       # rebuild the CPU image
MINERU_VARIANT=gpu ./scripts/build-mineru-image.sh  # Linux + NVIDIA only
```

Common first-run failures:
- `Connection refused`: start the container with `./scripts/run-mineru.sh`.
- HTTP `409`: the asynchronous MinerU task failed; inspect `./scripts/run-mineru.sh logs`.
- Slow first extraction: model files are downloading into the `mineru-models` volume.
- Language validation error: the pipeline backend accepts its own language set; Latin documents use `lang="ch"` in this integration.

For production benchmarking, run the GPU image on Linux and compare text order, formula extraction, and table structure on Dr. DocBench samples.